In [ ]:
# ReadME: Just run each section (I use Shift + Enter) to generate the output which is used to answer the questions in this assignment.
# Comments are sprinkled throughout the entirety of this project to help the reader (and also myself) better understand the code.
# oh also, IMPORTANT: run all sections in order otherwise certain sections may not work!!!

##### ### The University of Melbourne, School of Computing and Information Systems
# COMP30027 Machine Learning, 2025 Semester 1

## Assignment 1: Scam detection with naive Bayes


**Student ID(s):**     `1461238`

This iPython notebook is a template which you will use for your Assignment 1 submission.

**NOTE: YOU SHOULD ADD YOUR RESULTS, GRAPHS, AND FIGURES FROM YOUR OBSERVATIONS IN THIS FILE TO YOUR REPORT (the PDF file).** Results, figures, etc. which appear in this file but are NOT included in your report will not be marked.

**Adding proper comments to your code is MANDATORY. **

## 1. Supervised model training


In [3]:
# COMP30027 Project 1 - Scam Detection with naive bayes
# Just Run the Code for the results

import pandas as pd

training_dataset = pd.read_csv("sms_supervised_train.csv") # read the csv file

word_list = [] # create a list where all the vocabulary will be stored
# iterate through each seperate message in the textPreprocessed column
for sms in training_dataset['textPreprocessed']:
    # convert the float to a string and split it into separate words
    sms = str(sms)
    words = sms.split()
    # add each word into the word list
    for word in words:
        word_list.append(word)
word_list = set(word_list) #remove duplicates
word_list = list(word_list) #convert back into a list

# create a vocabulary index which maps each word to the column number in the count matrix
vocab_index = {}
for index, word in enumerate(word_list):
    vocab_index[word] = index
    
num_messages = len(training_dataset)
num_vocab = len(word_list)

# now create the count matrix, for now with only 'zero' values
count_matrix = []

for value in range(num_messages):
    row = [0] * num_vocab
    count_matrix.append(row)

# fill the matrix
for i, sms in enumerate(training_dataset['textPreprocessed']):
    sms = str(sms)
    for word in sms.split():
        if word in vocab_index:
            word_index = vocab_index[word]
            count_matrix[i][word_index] += 1

# count how many messages in each class
count_0 = 0
count_1 = 0
for value in training_dataset['class']:
    if value == 0:
        count_0 += 1
    elif value == 1:
        count_1 += 1

# compute prior probabilities and store in a dictionary 
prior_probabilities = {0 : count_0 / num_messages , 1 : count_1 / num_messages}

# now use Laplace smoothing to compute P(word | class) for both classes

alpha = 1 # this is the Laplace smoothing parameter

# initliase word count arrays for both the classes first

word_count_0 = [0] * num_vocab 
word_count_1 = [0] * num_vocab

# then we count the word occurrences in each class
for i, sms in enumerate(training_dataset['textPreprocessed']):
    sms = str(sms)
    words = sms.split()
    label = training_dataset['class'].iloc[i]

    for word in words:
        if word in vocab_index:
            word_index = vocab_index[word]
            if label == 0:
                word_count_0[word_index] += 1
            elif label == 1:
                word_count_1[word_index] += 1

# then we can get the total number of words in each class using sum
total_words_0 = sum(word_count_0)
total_words_1 = sum(word_count_1)

# initialise arrays for likelihoods
likelihood_0 = [0] * num_vocab
likelihood_1 = [0] * num_vocab

# calculate likelihoods using Laplace smoothing

for i in range(num_vocab):
    likelihood_0[i] = (word_count_0[i] + alpha) / (total_words_0 + alpha * num_vocab)
    likelihood_1[i] = (word_count_1[i] + alpha) / (total_words_1 + alpha * num_vocab)

# now we compute the ratio for every word

predictive_ratios = []

for i in range(num_vocab):
    word = word_list[i]
    ratio = likelihood_1[i] / likelihood_0[i]
    predictive_ratios.append((word,ratio))

# print prior probabilites of both classes
print("P(class = 0) = ", prior_probabilities[0])
print("P(class = 1) = ", prior_probabilities[1], "\n")

# compute and print the top 10 most probable words in each class + their probability values

word_to_likelihood_0 = list(zip(word_list, likelihood_0))
word_to_likelihood_1 = list(zip(word_list, likelihood_1))

word_to_likelihood_0 = sorted(word_to_likelihood_0, key = lambda x: x[1], reverse = True)
word_to_likelihood_1 = sorted(word_to_likelihood_1, key = lambda x: x[1], reverse = True)

top_10_0 = word_to_likelihood_0[:10]
top_10_1 = word_to_likelihood_1[:10]

print("Top 10 most probable words in Class 0 (legit):")
for word, prob in top_10_0:
    print(word + " " + str(round(prob, 4)))
print("Top 10 most probable words in Class 1 (scam):")
for word, prob in top_10_1:
    print(word + " " + str(round(prob, 4)))

# compute and print top 10 most predictive words for each class + their probability ratios

top_10_legit = sorted(predictive_ratios, key = lambda x: x[1])
top_10_scam = sorted(predictive_ratios, key = lambda x: x[1], reverse = True)

top_10_legit = top_10_legit[:10]
top_10_scam = top_10_scam[:10]

print("\n Top 10 most predictive legitimate words:")
for word, ratio in top_10_legit:
    print(word + " " + str(round(ratio, 4)))
print("Top 10 most predictive scam words:")
for word, ratio in top_10_scam:
    print(word + " " + str(round(ratio, 4)))

P(class = 0) =  0.8
P(class = 1) =  0.2 

Top 10 most probable words in Class 0 (legit):
. 0.0793
, 0.026
? 0.0256
u 0.0189
... 0.0187
! 0.0172
.. 0.0149
; 0.0132
& 0.0131
go 0.0111
Top 10 most probable words in Class 1 (scam):
. 0.0565
! 0.0243
, 0.0235
call 0.0205
£ 0.0139
free 0.0105
/ 0.0091
2 0.0088
& 0.0087
? 0.0085

 Top 10 most predictive legitimate words:
; 0.0165
... 0.0174
gt 0.0185
lt 0.0187
:) 0.0209
ü 0.0313
lor 0.0347
ok 0.0405
hope 0.0405
d 0.0474
Top 10 most predictive scam words:
prize 99.0509
tone 64.0917
£ 49.7197
select 46.6122
claim 45.9648
paytm 36.9013
code 34.9591
award 32.0459
won 31.0748
18 29.1326


## 2. Supervised model evaluation

In [13]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, classification_report
import numpy as np

test_dataset = pd.read_csv("sms_test.csv") # load the test dataset

training_texts = training_dataset['textPreprocessed']
training_labels = training_dataset['class']

training_texts = training_texts.fillna("").astype(str) # this ensures there's no null values, that they're all strings

vectorizer = CountVectorizer() # create a countvectorizer

training_texts_counts_matrix = vectorizer.fit_transform(training_texts) # use vectoriser to transform training texts into a count matrix

# now initialise + train the naive bayes model
nb_classifier = MultinomialNB()
nb_classifier.fit(training_texts_counts_matrix, training_labels)

# prepare the test texts
test_texts = test_dataset["textPreprocessed"]
test_counts_matrix = vectorizer.transform(test_texts)

predicted_classes = nb_classifier.predict(test_counts_matrix) # make predictions on the text test

test_dataset["predicted_class"] = predicted_classes # add these predictions to test dataset

true_test_labels = test_dataset["class"] # now extract true test labels

# print confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(true_test_labels, predicted_classes))

# calculate accuracy, precision and recall
accuracy = accuracy_score(true_test_labels, predicted_classes)
precision = precision_score(true_test_labels, predicted_classes, average=None)
recall = recall_score(true_test_labels, predicted_classes, average=None)

print("\nAccuracy:", round(accuracy, 4))

# print precision and recall
for i, class_label in enumerate(nb_classifier.classes_):
    print(f"Class {class_label} - Precision: {round(precision[i], 4)}, Recall: {round(recall[i], 4)}")

# print the classification report
print("\nFull Classification Report:")
print(classification_report(true_test_labels, predicted_classes))

# Question 2 in the Assignment also asks for examples of test instances (+ R values) which are:
# 1. Classified as scam with high confidence
# 2. Classified as non-malicious with high confidence
# 3. On the boundary between the two classes
# This next part of the code addresses this requirement

predicted_probabilities = nb_classifier.predict_proba(test_counts_matrix) # compute predicted probabilities
r_values = predicted_probabilities[:, 1] / predicted_probabilities[:, 0] # computer r values using formula in figure 8 of ass specification

sorted_r_values = np.argsort(r_values) # sort r-values in ascending order
highest_r_values = sorted_r_values[-3:][::-1] # get the 3 highest
lowest_r_values = sorted_r_values[:3] # get the 3 lowest
nearest_to_1 = np.argsort(np.abs(r_values - 1))[:3] # get the 3 closest to 1

# create a function for the printing of messages to reduce repetitiveness
def print_function(text, indices):
    print(text)
    for i in indices:
        print("Message: " + test_dataset['textPreprocessed'][i])
        print("R = {:.3f}".format(r_values[i]))
        print()

# now just print and voila...
print_function("Top 3 high-confidence scam predictions:", highest_r_values)
print_function("Top 3 high-confidence non-malicious predictions:", lowest_r_values)
print_function("Top 3 most 'on the boundary' predictions:", nearest_to_1)

Confusion Matrix:
[[788  12]
 [ 18 182]]

Accuracy: 0.97
Class 0 - Precision: 0.9777, Recall: 0.985
Class 1 - Precision: 0.9381, Recall: 0.91

Full Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       800
           1       0.94      0.91      0.92       200

    accuracy                           0.97      1000
   macro avg       0.96      0.95      0.95      1000
weighted avg       0.97      0.97      0.97      1000

Top 3 high-confidence scam predictions:
Message: . please call award apply end rs. vodafone todays number select receive match quoting claim code standard rate 2,00,000 6299257179
R = 77060675900528361472.000

Message: . please call award apply end rs. vodafone todays number select receive match quoting claim code standard rate 2,00,000 7908807538
R = 57795506925395927040.000

Message: . 4 + ! call * 150 holiday & urgent 18 t landline cash cs await collection po box sae complimentary ppm 10,000 ib

## 3. Extending the model with semi-supervised training

In [29]:
# So, very simply, what's being done in this part is:
# a. use the earlier model to predict labels for the unlabelled data - we can treat these predictions are true
# b. combine this new, now-labelled data with the original data in order to re-train the model

# load the unlabelled data
unlabelled_dataset = pd.read_csv("sms_unlabelled.csv")
unlabelled_dataset['textPreprocessed'] = unlabelled_dataset['textPreprocessed'].fillna("").astype(str)

# transform it into a vector form
unlabelled_texts_counts_matrix = vectorizer.transform(unlabelled_dataset['textPreprocessed'])

class_probs_unlabelled = nb_classifier.predict_proba(unlabelled_texts_counts_matrix) # gives the models probability/confidence for both classes
predicted_labels_unlabelled = nb_classifier.predict(unlabelled_texts_counts_matrix) # gives the predicted class

# we only want to use instances where the model predicts with high confidence - more on this in my write-up!

# add two new columns to the unlabelled dataset; "predicted_label" and "prediction_confidence"
# self-explanatory what each of these columns show
unlabelled_dataset = unlabelled_dataset.assign(
    predicted_label = predicted_labels_unlabelled,
    prediction_confidence = class_probs_unlabelled.max(axis=1)
)

# filter for high confidence rows, i'll use predictions above 0.85 confidence - more on my reasoning in my write-up!
high_confidence_data = unlabelled_dataset[unlabelled_dataset['prediction_confidence'] > 0.85]

# just extract these columns of interest
high_confidence_texts = high_confidence_data['textPreprocessed']
high_confidence_labels = high_confidence_data['predicted_label']

# now create a new dataframe that stores the high confidence predicted data
labelled_dataset = pd.DataFrame({
    'textPreprocessed': high_confidence_texts.values,
    'class': high_confidence_labels
})

# combine the labelled data with the original data
combined_dataset = pd.concat([training_dataset, labelled_dataset], ignore_index=True)
combined_dataset['textPreprocessed'] = combined_dataset['textPreprocessed'].fillna("").astype(str) # ensure there's no null values

# prepare this combined data so the model can learn from it
new_vectorizer = CountVectorizer()
combined_texts_counts_matrix = new_vectorizer.fit_transform(combined_dataset['textPreprocessed'])
combined_labels = combined_dataset['class']

# re-train the model
semi_supervised_classifier = MultinomialNB()
semi_supervised_classifier.fit(combined_texts_counts_matrix, combined_labels)

# Alright that's the label propagation process done. Now let me address the part of the question asking for evaluation.
# I can do this by just testing the model on a held-out validation set from the original training data.

from sklearn.model_selection import train_test_split

# split the original labelled data, 80% for training and 20% for testing
training_data_subset, validation_subset = train_test_split(training_dataset, test_size=0.2, random_state=69)

# now recombine the 80% training portion with the labelled data
combined_dataset = pd.concat([training_data_subset, labelled_dataset], ignore_index=True)
combined_dataset['textPreprocessed'] = combined_dataset['textPreprocessed'].fillna("").astype(str) # no null values!

# vectorise and then re-train model
new_vectorizer = CountVectorizer()
combined_texts_counts_matrix = new_vectorizer.fit_transform(combined_dataset['textPreprocessed'])
combined_labels = combined_dataset['class']
semi_supervised_classifier = MultinomialNB()
semi_supervised_classifier.fit(combined_texts_counts_matrix, combined_labels)

# transform validation set into vector form, then make predictions
validation_texts_counts_matrix = new_vectorizer.transform(validation_subset['textPreprocessed'].fillna("").astype(str))
validation_labels = validation_subset['class']
validation_predictions = semi_supervised_classifier.predict(validation_texts_counts_matrix)

# print some results for evaluation
print("Evaluation for Validation Set")
print("Accuracy:", round(accuracy_score(validation_labels, validation_predictions),4))
print("Precision:", round(precision_score(validation_labels, validation_predictions),4))
print("Recall:", round(recall_score(validation_labels, validation_predictions),4))

# show how many labelled examples were added
print("Labelled messages added:", len(labelled_dataset))

Evaluation for Validation Set
Accuracy: 0.955
Precision: 0.8675
Recall: 0.9114
Labelled messages added: 1851


## 4. Semi-supervised model evaluation

In [37]:
# vectorise the test messages
test_texts_counts_matrix = new_vectorizer.transform(test_dataset['textPreprocessed'])
true_test_labels = test_dataset['class']

# make predictions
test_predictions = semi_supervised_classifier.predict(test_texts_counts_matrix)

# print evaluation metrics
print("Evaluation for Test Set (Semi-Supervised model)")
print("Accuracy:", round(accuracy_score(true_test_labels, test_predictions),4))
print("Precision:", round(precision_score(true_test_labels, test_predictions),4))
print("Recall:", round(recall_score(true_test_labels, test_predictions),4))

# print classification report innit
print("\nClassification Report:")
print(classification_report(true_test_labels, test_predictions))

Evaluation for Test Set (Semi-Supervised model)
Accuracy: 0.973
Precision: 0.9482
Recall: 0.915

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       800
           1       0.95      0.92      0.93       200

    accuracy                           0.97      1000
   macro avg       0.96      0.95      0.96      1000
weighted avg       0.97      0.97      0.97      1000

